# PrMers CUDA/OpenCL automatic selector — Kaggle T4×2 smoke test

Use Kaggle **Settings → Accelerator → GPU T4 x2**. This notebook builds from current PrMers sources inside Kaggle (native glibc compatibility), then tests AUTO and OpenCL across PRP, LL, P-1, ECM and Gaussian P-1 on both GPUs.

**No credentials, Turso/GMNet requests or campaign jobs are used.** This is a bounded smoke test, not a complete correctness/performance certification. Source defaults to `main`; the exact SHA is printed.


In [ ]:
import os, subprocess, pathlib
def check(cmd, cwd=None, timeout=1800):
    print('+', ' '.join(map(str,cmd)), flush=True)
    p=subprocess.run(cmd, cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=timeout)
    print(p.stdout[-12000:])
    if p.returncode: raise RuntimeError(f'command returned {p.returncode}')
    return p.stdout
check(['nvidia-smi','--query-gpu=index,uuid,name','--format=csv,noheader'])
check(['apt-get','update','-qq'])
check(['apt-get','install','-y','-qq','--no-install-recommends','git','make','g++','libgmp-dev','ocl-icd-opencl-dev','clinfo'])
root=pathlib.Path('/kaggle/working/PrMers')
if not root.exists():
    check(['git','clone','--depth','1','--branch','main','https://github.com/cherubrock-seb/PrMers.git',str(root)])
print('PrMers SHA:',check(['git','rev-parse','HEAD'],cwd=root).strip())
check(['python3','scripts/kaggle_dual_backend_smoke.py','--devices','0,1','--max-seconds','45'],cwd=root,timeout=3600)
